# <center> Make adata

## <center> Description

- env: squidpy_v06

## <center> Set-up

#### **libs**

In [1]:
import pandas as pd
import anndata as ad
from scipy import sparse
## helper functions
from workflow.scripts.funs.helper_funs import get_project_root

#### **paths**

In [2]:
from pathlib import Path

if "snakemake" in globals():
    # in_metadata = Path(snakemake.input["in_metadata"])
    # out_metadata = Path(snakemake.output["out_metadata"])
    print('Hello World')
else:
    ## get paths
    ROOT = get_project_root(globals())
    ## input
    in_metadata = ROOT / "output/preprocessing/objects/clean_metadata/metadata.tsv"
    in_counts = ROOT / "data/gene_counts.txt"
    in_genesymbols = ROOT / "output/preprocessing/objects/get_gene_symbols/genesymbols.tsv"
    ## output
    out_adata = (ROOT / "output/preprocessing/objects/make_adata/adata.h5ad")
    out_adata.parent.mkdir(parents=True, exist_ok=True)

## <center> Main

#### **Load data**

In [3]:
# clean metadata table
meta = pd.read_csv(in_metadata, sep='\t', index_col=0).set_index('sample_id')

meta.head()

,disease_group,QC_group,condition_group
sample_id,,,
SD1,HR_case,A,IFNG_neg
SD3,HR_case,A,IFNG_neg
SD5,HR_case,A,IFNG_neg
SD7,HR_case,A,IFNG_neg
SD9,HR_case,A,IFNG_neg


#### **Load count matrix**

In [4]:
# count matrix

counts = pd.read_csv(in_counts,sep='\t').set_index('Gene')
counts = counts.T

counts.head()

Gene,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000289624,ENSG00000289626,ENSG00000289629,ENSG00000289631,ENSG00000289632,ENSG00000289635,ENSG00000289637,ENSG00000289639,ENSG00000289640,ENSG00000289643
SD1,1332.000,0.0,996.306,127.251,45.7492,0.0,25.000,2727.99,523.0,1195.000,...,4.0,3.0,0.0,4.0,0.0,0.0,2.0,0.0,0.0,1.0
SD10,1088.000,0.0,1208.960,315.508,75.4915,0.0,1000.000,1931.98,410.0,955.002,...,11.0,1.0,1.0,2.0,0.0,0.0,2.0,0.0,0.0,2.0
SD11,1044.000,15.0,935.001,186.220,101.7800,1.0,171.000,1920.99,357.0,1506.000,...,0.0,5.0,5.0,2.0,0.0,0.0,0.0,0.0,0.0,6.0
SD12,994.004,17.0,987.931,211.926,132.0740,1.0,651.991,1927.99,397.0,1577.960,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0
SD13,2312.000,0.0,989.000,160.471,45.5289,1.0,48.000,2036.99,413.0,1215.000,...,17.0,2.0,0.0,5.0,0.0,2.0,0.0,0.0,0.0,0.0


In [5]:
# conversion table for gene symbols

genesymbols = pd.read_csv(in_genesymbols,sep='\t',index_col=0)

genesymbols.head()

,gene_symbol,gene_biotype,chromosome
gene_id,,,
ENSG00000210049,MT-TF,Mt_tRNA,MT
ENSG00000211459,MT-RNR1,Mt_rRNA,MT
ENSG00000210077,MT-TV,Mt_tRNA,MT
ENSG00000210082,MT-RNR2,Mt_rRNA,MT
ENSG00000209082,MT-TL1,Mt_tRNA,MT


#### **Make adata**

In [6]:
# align metadata and counts

sample_ids = meta.index[meta.index.isin(counts.index)]
counts = counts.reindex(index=sample_ids)
meta = meta.reindex(index=sample_ids)

all(meta.index == counts.index)

True

In [7]:
# make adata

adata = ad.AnnData(
    X=sparse.csr_matrix(counts.astype("float32")),
    obs=meta,
    var=pd.DataFrame(index=counts.columns),
)

adata.obs_names.name = "sample_id"
adata.var_names.name = "gene_id"

adata


AnnData object with n_obs × n_vars = 66 × 30155
    obs: 'disease_group', 'QC_group', 'condition_group'

In [8]:
# add genesymbols to .var

adata.var['genesymbol']=adata.var.index.map( genesymbols['gene_symbol'].to_dict() )

adata.var

,genesymbol
gene_id,
ENSG00000000003,TSPAN6
ENSG00000000005,TNMD
ENSG00000000419,DPM1
ENSG00000000457,SCYL3
ENSG00000000460,FIRRM
...,...
ENSG00000289635,NaN
ENSG00000289637,NaN
ENSG00000289639,NaN


In [9]:
# columns to correct datatype

## obs
adata.obs['disease_group']=adata.obs['disease_group'].astype('category').cat.reorder_categories(['HR_control','HR_case'],ordered=True)
adata.obs['QC_group']=adata.obs['QC_group'].astype('category').cat.reorder_categories(['A','B'],ordered=True)
adata.obs['condition_group']=adata.obs['condition_group'].astype('category').cat.reorder_categories(['IFNG_neg','IFNG_pos'],ordered=True)

#### **Save**

In [10]:
adata.write(out_adata)